# Customer Catalogue GP Homologation

## Step 1 - Load, validate and normalize Global Parents

This notebook:

- Loads the customer catalogue
- Validates required columns
- Counts unique Ship To Numbers per Global Parent
- Creates a normalized GP name
- Exports a summary for review

The original catalogue is NEVER modified.

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path
from rapidfuzz import process
from rapidfuzz import fuzz

In [178]:
# ==========================================
# Configuration
# ==========================================

INPUT_FILE = r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\July 2026\Industrial Market\Industrial_Market.csv'

OUTPUT_FOLDER = "output"

GP_COLUMN = "Global Parent"

SHIP_TO_COLUMN = "Ship To Number"

REMOVE_ACCENTS = True
REMOVE_PUNCTUATION = True
REMOVE_EXTRA_SPACES = True
REMOVE_LEGAL_SUFFIXES = True

In [179]:
LEGAL_SUFFIXES = [

    "INCORPORATED",
    "INC",
    "CORPORATION",
    "CORP",
    "COMPANY",
    "CO",
    "LIMITED",
    "LTD",
    "LLC",
    "LLP",
    "PLC",
    "LP",
    "GMBH",
    "AG",
    "BV",
    "NV",
    "SA",
    "SAS",
    "SPA",
    "SRL",
    "PTY",
    "PVT",
    "KK"

]

In [180]:
def normalize_gp(name):

    if pd.isna(name):
        return ""

    text = str(name).upper()

    if REMOVE_ACCENTS:
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))

    if REMOVE_PUNCTUATION:
        text = re.sub(r"[^\w\s]", " ", text)

    if REMOVE_LEGAL_SUFFIXES:

        words = text.split()

        words = [
            word
            for word in words
            if word not in LEGAL_SUFFIXES
        ]

        text = " ".join(words)

    if REMOVE_EXTRA_SPACES:
        text = re.sub(r"\s+", " ", text).strip()

    return text

In [181]:
df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded: {len(df):,}")

Rows loaded: 49,493


In [182]:
required_columns = [

    GP_COLUMN,
    SHIP_TO_COLUMN

]

missing = [

    column
    for column in required_columns
    if column not in df.columns

]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )

print("All required columns found.")

All required columns found.


In [183]:
df["Normalized GP"] = df[GP_COLUMN].apply(normalize_gp)

df["Normalization Changed"] = (
    df[GP_COLUMN].str.upper().fillna("")
    !=
    df["Normalized GP"]
)

In [184]:
gp_summary = (

    df.groupby(GP_COLUMN)
      .agg(

          Ship_To_Count=(
              SHIP_TO_COLUMN,
              "nunique"
          ),

          Row_Count=(
              SHIP_TO_COLUMN,
              "count"
          ),

          GP_Country=(
              "GP Country",
              "first"
          ),

          Normalized_GP=(
              "Normalized GP",
              "first"
          ),

          Normalization_Changed=(
              "Normalization Changed",
              "first")

      )

      .reset_index()

)

In [185]:
gp_summary = gp_summary.sort_values(

    by=[
        "Ship_To_Count",
        GP_COLUMN
    ],

    ascending=[
        False,
        True
    ]

)

In [186]:
gp_summary.head(20)

,Global Parent,Ship_To_Count,Row_Count,GP_Country,Normalized_GP,Normalization_Changed
16887,TOTAL,121,272,France,TOTAL,False
14566,SAMPLE ACCOUNT,105,185,NaN,SAMPLE ACCOUNT,False
316,ADJUSTMENT,94,202,England,ADJUSTMENT,False
5561,FASTENAL,94,178,United States Of America,FASTENAL,False
12565,PARKER HANNIFIN,87,151,United States Of America,PARKER HANNIFIN,False
9978,Linde,76,145,Ireland,LINDE,False
2078,BODYCOTE,70,131,England,BODYCOTE,False
4693,EATON,66,133,United States Of America,EATON,False
12693,PENINSULAR DE LUBRICANTES,65,70,Spain,PENINSULAR DE LUBRICANTES,False
3967,DANFOSS,50,100,Denmark,DANFOSS,False


In [187]:
Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

output_file = Path(OUTPUT_FOLDER) / "01_gp_summary.csv"

gp_summary.to_csv(

    output_file,

    index=False

)

print(f"Saved to: {output_file}")

Saved to: output\01_gp_summary.csv


In [188]:
exact_groups = (
    gp_summary
    .groupby("Normalized_GP")
    .agg(
        Variants=(GP_COLUMN, list),
        Number_of_Variants=(GP_COLUMN, "nunique"),
        Total_Ship_To_Count=("Ship_To_Count", "sum"),
        Total_Row_Count=("Row_Count", "sum"),
        Countries=("GP_Country", lambda x: sorted(set(x.dropna()))),
        Any_Normalization_Changed=("Normalization_Changed", "any")
    )
    .reset_index()
)

In [189]:
exact_groups = exact_groups[
    exact_groups["Number_of_Variants"] > 1
].copy()

In [190]:
exact_groups = exact_groups.sort_values(
    by=[
        "Total_Ship_To_Count",
        "Number_of_Variants"
    ],
    ascending=False
)

In [191]:
exact_groups.head(20)

,Normalized_GP,Variants,Number_of_Variants,Total_Ship_To_Count,Total_Row_Count,Countries,Any_Normalization_Changed
6170,GENUINE PARTS,"[GENUINE PARTS, GENUINE PARTS COMPANY]",2,20,43,[United States Of America],True
12361,PENTAIR,"[PENTAIR, Pentair]",2,17,27,[United States Of America],False
10910,MOLEX,"[MOLEX INC, MOLEX]",2,16,27,[United States Of America],True
16063,TENNECO,"[TENNECO, Tenneco]",2,12,20,[United States Of America],False
635,ALFA LAVAL,"[ALFA LAVAL, ALFA LAVAL INC]",2,11,21,[Sweden],True
8450,JOHNSON CONTROLS,"[JOHNSON CONTROLS INC, Johnson Controls]",2,10,23,[United States Of America],True
8634,KAMAN,"[Kaman Corporation, KAMAN]",2,10,19,[United States Of America],True
14284,SCHAEFFLER,"[SCHAEFFLER, Schaeffler]",2,10,16,[Germany],False
3415,CONDO,"[CONDO, CONDO INC, Condo Inc]",3,9,11,[United States Of America],True
10281,MECALUX,"[MECALUX, MECALUX SA]",2,9,18,[Spain],True


In [192]:
output_file = Path(OUTPUT_FOLDER) / "02_exact_normalized_groups.csv"

exact_groups.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")

Saved to: output\02_exact_normalized_groups.csv


In [193]:
entities = exact_groups.copy()

entities = entities.rename(
    columns={
        "Normalized_GP": "Representative",
        "Variants": "Members",
        "Total_Ship_To_Count": "Ship_To_Count",
        "Total_Row_Count": "Row_Count",
        "Countries": "Countries"
    }
)

In [194]:
entities.insert(
    0,
    "Entity_ID",
    [
        f"E{i:06d}"
        for i in range(1, len(entities) + 1)
    ]
)

In [195]:
grouped_gps = set()

for members in entities["Members"]:
    grouped_gps.update(members)

singletons = gp_summary[
    ~gp_summary[GP_COLUMN].isin(grouped_gps)
].copy()

In [196]:
singletons["Representative"] = singletons["Normalized_GP"]

singletons["Members"] = singletons[GP_COLUMN]

singletons["Countries"] = singletons["GP_Country"]


start = len(entities) + 1

singletons.insert(
    0,
    "Entity_ID",
    [
        f"E{i:06d}"
        for i in range(start, start + len(singletons))
    ]
)

In [197]:
singletons = singletons[
    [
        "Entity_ID",
        "Representative",
        "Members",
        "Ship_To_Count",
        "Row_Count",
        "Countries"
    ]
]

In [198]:
entities = pd.concat(
    [
        entities,
        singletons
    ],
    ignore_index=True
)

In [199]:
entities = entities.sort_values(
    by="Ship_To_Count",
    ascending=False
)

In [200]:
entities.head(20)

,Entity_ID,Representative,Members,Number_of_Variants,Ship_To_Count,Row_Count,Countries,Any_Normalization_Changed
561,E000562,TOTAL,TOTAL,NaN,121,272,France,NaN
562,E000563,SAMPLE ACCOUNT,SAMPLE ACCOUNT,NaN,105,185,NaN,NaN
563,E000564,ADJUSTMENT,ADJUSTMENT,NaN,94,202,England,NaN
564,E000565,FASTENAL,FASTENAL,NaN,94,178,United States Of America,NaN
565,E000566,PARKER HANNIFIN,PARKER HANNIFIN,NaN,87,151,United States Of America,NaN
566,E000567,LINDE,Linde,NaN,76,145,Ireland,NaN
567,E000568,BODYCOTE,BODYCOTE,NaN,70,131,England,NaN
568,E000569,EATON,EATON,NaN,66,133,United States Of America,NaN
569,E000570,PENINSULAR DE LUBRICANTES,PENINSULAR DE LUBRICANTES,NaN,65,70,Spain,NaN
570,E000571,DANFOSS,DANFOSS,NaN,50,100,Denmark,NaN


In [201]:
def list_to_string(value):

    if isinstance(value, list):
        return " | ".join(value)

    if pd.isna(value):
        return ""

    return str(value)

entities["Members"] = entities["Members"].apply(list_to_string)

entities["Countries"] = entities["Countries"].apply(list_to_string)


output_file = Path(OUTPUT_FOLDER) / "03_entities.csv"
entities.to_csv(
    output_file,
    index=False
)

print(f"Saved to: {output_file}")

Saved to: output\03_entities.csv


In [ ]:
# ===================================
# Fuzzy Matching Configuration
# ===================================

TOP_CANDIDATES = 10

MIN_SCORE = 95

SCORER = fuzz.token_sort_ratio

In [ ]:
entity_names = entities["Representative"].tolist()

In [ ]:
def generate_candidates(name, choices):

    return process.extract(

        query=name,

        choices=choices,

        scorer=SCORER,

        limit=TOP_CANDIDATES

    )

In [ ]:
generate_candidates(

    "ABB",

    entity_names

)

In [ ]:
def generate_candidates(name):

    return process.extract(

        query=name,

        choices=entity_names,

        scorer=SCORER,

        limit=TOP_CANDIDATES

    )

In [ ]:
def calculate_scores(name1, name2):

    return {

        "Ratio": round(fuzz.ratio(name1, name2), 1),

        "Partial_Ratio": round(fuzz.partial_ratio(name1, name2), 1),

        "Token_Sort": round(fuzz.token_sort_ratio(name1, name2), 1),

        "Token_Set": round(fuzz.token_set_ratio(name1, name2), 1)

    }

In [ ]:
def is_high_confidence(scores):

    return (

        scores["Token_Set"] >= MIN_SCORE

        or

        scores["Token_Sort"] >= MIN_SCORE

    )

In [ ]:
name1 = "ABB"

name2 = "ABB GROUP"

scores = calculate_scores(name1, name2)

print(scores)

print(is_high_confidence(scores))

In [ ]:
entity_lookup = (
    entities
    .set_index("Representative")
    .to_dict("index")
)

In [ ]:
fuzzy_matches = []

In [ ]:
for _, entity in entities.iterrows():

    representative = entity["Representative"]

    candidates = generate_candidates(representative)

    for candidate_name, _, _ in candidates:

        # Ignore itself
        if candidate_name == representative:
            continue

        scores = calculate_scores(
            representative,
            candidate_name
        )

        if not is_high_confidence(scores):
            continue

        candidate = entity_lookup[candidate_name]

        fuzzy_matches.append({

            "Entity_1": entity["Entity_ID"],
            "Representative_1": representative,
            "Ship_To_1": entity["Ship_To_Count"],

            "Entity_2": candidate["Entity_ID"],
            "Representative_2": candidate_name,
            "Ship_To_2": candidate["Ship_To_Count"],

            **scores

        })

In [ ]:
fuzzy_matches = pd.DataFrame(fuzzy_matches)

In [ ]:
fuzzy_matches["Pair"] = fuzzy_matches.apply(

    lambda row: tuple(sorted([
        row["Entity_1"],
        row["Entity_2"]
    ])),

    axis=1
)

fuzzy_matches = (
    fuzzy_matches
    .drop_duplicates("Pair")
    .drop(columns="Pair")
)

In [ ]:
fuzzy_matches = fuzzy_matches.sort_values(

    by=[
        "Token_Set",
        "Ship_To_1",
        "Ship_To_2"
    ],

    ascending=False
)

In [ ]:
fuzzy_matches.head(20)

Temporal cells

In [202]:
print(len(exact_groups))

561


In [203]:
exact_groups.head(20)

,Normalized_GP,Variants,Number_of_Variants,Total_Ship_To_Count,Total_Row_Count,Countries,Any_Normalization_Changed
6170,GENUINE PARTS,"[GENUINE PARTS, GENUINE PARTS COMPANY]",2,20,43,[United States Of America],True
12361,PENTAIR,"[PENTAIR, Pentair]",2,17,27,[United States Of America],False
10910,MOLEX,"[MOLEX INC, MOLEX]",2,16,27,[United States Of America],True
16063,TENNECO,"[TENNECO, Tenneco]",2,12,20,[United States Of America],False
635,ALFA LAVAL,"[ALFA LAVAL, ALFA LAVAL INC]",2,11,21,[Sweden],True
8450,JOHNSON CONTROLS,"[JOHNSON CONTROLS INC, Johnson Controls]",2,10,23,[United States Of America],True
8634,KAMAN,"[Kaman Corporation, KAMAN]",2,10,19,[United States Of America],True
14284,SCHAEFFLER,"[SCHAEFFLER, Schaeffler]",2,10,16,[Germany],False
3415,CONDO,"[CONDO, CONDO INC, Condo Inc]",3,9,11,[United States Of America],True
10281,MECALUX,"[MECALUX, MECALUX SA]",2,9,18,[Spain],True


In [204]:
gp_summary['Normalization_Changed'].value_counts()

Normalization_Changed
False    15700
True      3080
Name: count, dtype: int64

In [205]:
print(len(entities))

18183


In [206]:
print(len(gp_summary))
print(len(grouped_gps))
print(len(singletons))

18780
1158
17622


In [207]:
print(df[GP_COLUMN].nunique())
print(len(gp_summary))

18780
18780
